# 🧪 Clase 1: Testing y Evaluación de LLMs

## Bienvenido a la Semana 4, Clase 1

En esta clase aprenderás:
- ✅ Por qué testing de LLMs es diferente
- ✅ Métricas de evaluación
- ✅ Testing de prompts
- ✅ Testing de RAG
- ✅ Testing de agentes
- ✅ Deployment best practices

---

In [ ]:
!pip install langchain langchain-openai pytest ragas deepeval -q

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser
import json

load_dotenv()
llm = ChatOpenAI(model="gpt-4", temperature=0)
print("✅ Configuración completada")

## 🤔 Parte 1: ¿Por qué Testing de LLMs es Diferente?

### Testing Tradicional vs Testing de LLMs

**Testing Tradicional**:
```python
assert suma(2, 2) == 4  # Determinístico
```

**Testing de LLMs**:
```python
respuesta = llm("¿Qué es Python?")
# ¿Cómo validar? La respuesta varía cada vez
```

### Desafíos

1. **No determinístico**: Misma entrada → diferentes salidas
2. **Subjetivo**: ¿Qué es una "buena" respuesta?
3. **Contexto**: Depende del contexto y uso
4. **Costo**: Cada test cuesta dinero (API calls)

### Soluciones

- ✅ Usar temperatura=0 para tests
- ✅ Evaluar múltiples aspectos
- ✅ Usar LLMs para evaluar LLMs
- ✅ Crear datasets de evaluación

## 📊 Parte 2: Métricas de Evaluación

### 1. Métricas Básicas

In [ ]:
def evaluar_longitud(respuesta: str, min_words: int = 10, max_words: int = 100) -> dict:
    """Evalúa la longitud de la respuesta."""
    palabras = len(respuesta.split())
    return {
        "palabras": palabras,
        "valido": min_words <= palabras <= max_words,
        "mensaje": f"{'✅' if min_words <= palabras <= max_words else '❌'} {palabras} palabras"
    }

def evaluar_formato(respuesta: str, debe_contener: list) -> dict:
    """Evalúa si la respuesta contiene elementos esperados."""
    encontrados = [item for item in debe_contener if item.lower() in respuesta.lower()]
    return {
        "encontrados": encontrados,
        "faltantes": [item for item in debe_contener if item not in encontrados],
        "score": len(encontrados) / len(debe_contener)
    }

# Probar
respuesta_test = "Python es un lenguaje de programación interpretado y de alto nivel."
print("📏 Evaluación de longitud:")
print(evaluar_longitud(respuesta_test))
print("\n📋 Evaluación de formato:")
print(evaluar_formato(respuesta_test, ["Python", "lenguaje", "programación"]))

### 2. LLM-as-a-Judge

Usar un LLM para evaluar otro LLM:

In [ ]:
def llm_judge(pregunta: str, respuesta: str, criterios: list) -> dict:
    """Usa un LLM para evaluar la calidad de una respuesta."""
    
    criterios_str = "\n".join([f"- {c}" for c in criterios])
    
    prompt = ChatPromptTemplate.from_template(
        """Evalúa esta respuesta según los criterios dados.

Pregunta: {pregunta}
Respuesta: {respuesta}

Criterios:
{criterios}

Da una puntuación de 1-10 para cada criterio y un comentario.
Formato JSON:
{{
    "criterio1": {{"score": X, "comentario": "..."}},
    "score_total": X
}}"""
    )
    
    chain = prompt | llm | StrOutputParser()
    resultado = chain.invoke({
        "pregunta": pregunta,
        "respuesta": respuesta,
        "criterios": criterios_str
    })
    
    try:
        return json.loads(resultado)
    except:
        return {"raw": resultado}

# Probar
pregunta = "¿Qué es machine learning?"
respuesta = "Machine learning es una rama de la IA donde las máquinas aprenden de datos."
criterios = ["Precisión", "Claridad", "Completitud"]

print("⚖️ Evaluación con LLM-as-a-Judge:\n")
evaluacion = llm_judge(pregunta, respuesta, criterios)
print(json.dumps(evaluacion, indent=2, ensure_ascii=False))

## 🧪 Parte 3: Testing de Prompts

In [ ]:
# Dataset de prueba
test_cases = [
    {
        "input": "Python",
        "expected_keywords": ["lenguaje", "programación"],
        "min_words": 15
    },
    {
        "input": "JavaScript",
        "expected_keywords": ["web", "navegador"],
        "min_words": 15
    },
    {
        "input": "SQL",
        "expected_keywords": ["base de datos", "consultas"],
        "min_words": 15
    }
]

def test_prompt(template: str, test_cases: list) -> dict:
    """Prueba un prompt con múltiples casos."""
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()
    
    resultados = []
    
    for i, caso in enumerate(test_cases, 1):
        print(f"\n🧪 Test {i}/{len(test_cases)}: {caso['input']}")
        
        respuesta = chain.invoke({"lenguaje": caso["input"]})
        
        # Evaluar
        eval_longitud = evaluar_longitud(respuesta, min_words=caso["min_words"])
        eval_formato = evaluar_formato(respuesta, caso["expected_keywords"])
        
        resultado = {
            "input": caso["input"],
            "respuesta": respuesta,
            "longitud_ok": eval_longitud["valido"],
            "keywords_score": eval_formato["score"],
            "passed": eval_longitud["valido"] and eval_formato["score"] >= 0.5
        }
        
        resultados.append(resultado)
        print(f"  Longitud: {eval_longitud['mensaje']}")
        print(f"  Keywords: {eval_formato['score']:.0%}")
        print(f"  {'✅ PASS' if resultado['passed'] else '❌ FAIL'}")
    
    # Resumen
    passed = sum(1 for r in resultados if r["passed"])
    total = len(resultados)
    
    return {
        "resultados": resultados,
        "passed": passed,
        "total": total,
        "success_rate": passed / total
    }

# Probar dos versiones de prompt
print("="*80)
print("PROMPT V1: Simple")
print("="*80)
template_v1 = "Explica qué es {lenguaje}"
results_v1 = test_prompt(template_v1, test_cases)

print("\n" + "="*80)
print("PROMPT V2: Detallado")
print("="*80)
template_v2 = "Explica qué es {lenguaje} en 2-3 oraciones. Incluye su propósito principal."
results_v2 = test_prompt(template_v2, test_cases)

print("\n" + "="*80)
print("COMPARACIÓN")
print("="*80)
print(f"V1 Success Rate: {results_v1['success_rate']:.0%}")
print(f"V2 Success Rate: {results_v2['success_rate']:.0%}")

## 🔍 Parte 4: Testing de RAG

In [ ]:
# Métricas específicas para RAG
def evaluar_rag(pregunta: str, respuesta: str, documentos_fuente: list) -> dict:
    """Evalúa un sistema RAG."""
    
    # 1. Relevancia: ¿La respuesta es relevante a la pregunta?
    prompt_relevancia = ChatPromptTemplate.from_template(
        """¿Esta respuesta es relevante para la pregunta?
Pregunta: {pregunta}
Respuesta: {respuesta}

Responde solo: SI o NO"""
    )
    
    # 2. Fidelidad: ¿La respuesta está basada en los documentos?
    prompt_fidelidad = ChatPromptTemplate.from_template(
        """¿Esta respuesta está basada en los documentos fuente?
Documentos: {documentos}
Respuesta: {respuesta}

Responde solo: SI o NO"""
    )
    
    chain = llm | StrOutputParser()
    
    relevancia = chain.invoke(prompt_relevancia.format(pregunta=pregunta, respuesta=respuesta))
    fidelidad = chain.invoke(prompt_fidelidad.format(
        documentos="\n".join(documentos_fuente),
        respuesta=respuesta
    ))
    
    return {
        "relevancia": "SI" in relevancia.upper(),
        "fidelidad": "SI" in fidelidad.upper(),
        "passed": "SI" in relevancia.upper() and "SI" in fidelidad.upper()
    }

# Ejemplo
pregunta = "¿Qué es Python?"
respuesta = "Python es un lenguaje de programación interpretado."
documentos = [
    "Python es un lenguaje de programación de alto nivel.",
    "Python fue creado por Guido van Rossum."
]

print("🔍 Evaluación de RAG:\n")
eval_rag = evaluar_rag(pregunta, respuesta, documentos)
print(f"Relevancia: {'✅' if eval_rag['relevancia'] else '❌'}")
print(f"Fidelidad: {'✅' if eval_rag['fidelidad'] else '❌'}")
print(f"\nResultado: {'✅ PASS' if eval_rag['passed'] else '❌ FAIL'}")

## 🤖 Parte 5: Testing de Agentes

In [ ]:
def test_agent_behavior(agent_executor, test_scenarios: list) -> dict:
    """Prueba el comportamiento de un agente."""
    resultados = []
    
    for scenario in test_scenarios:
        print(f"\n🧪 Escenario: {scenario['name']}")
        print(f"Input: {scenario['input']}")
        
        try:
            # Ejecutar agente
            result = agent_executor.invoke({"input": scenario["input"]})
            output = result.get("output", "")
            
            # Verificar herramientas usadas
            tools_used = scenario.get("expected_tools", [])
            tools_check = all(tool in str(result) for tool in tools_used)
            
            # Verificar output
            output_check = any(keyword in output.lower() for keyword in scenario.get("expected_keywords", []))
            
            passed = tools_check and output_check
            
            resultado = {
                "scenario": scenario["name"],
                "output": output,
                "tools_check": tools_check,
                "output_check": output_check,
                "passed": passed
            }
            
            print(f"Tools: {'✅' if tools_check else '❌'}")
            print(f"Output: {'✅' if output_check else '❌'}")
            print(f"{'✅ PASS' if passed else '❌ FAIL'}")
            
        except Exception as e:
            resultado = {
                "scenario": scenario["name"],
                "error": str(e),
                "passed": False
            }
            print(f"❌ ERROR: {e}")
        
        resultados.append(resultado)
    
    passed = sum(1 for r in resultados if r["passed"])
    return {
        "resultados": resultados,
        "passed": passed,
        "total": len(resultados),
        "success_rate": passed / len(resultados)
    }

print("✅ Función de testing de agentes definida")

## 🚀 Parte 6: Deployment Best Practices

### Checklist Pre-Deployment

#### 1. Testing
- ✅ Tests unitarios pasando
- ✅ Tests de integración
- ✅ Evaluación con dataset de prueba
- ✅ Testing de edge cases

#### 2. Monitoreo
- ✅ LangSmith configurado
- ✅ Logging implementado
- ✅ Alertas configuradas
- ✅ Métricas de uso

#### 3. Seguridad
- ✅ API keys en variables de entorno
- ✅ Rate limiting
- ✅ Input validation
- ✅ Output filtering

#### 4. Performance
- ✅ Caching implementado
- ✅ Timeouts configurados
- ✅ Retry logic
- ✅ Optimización de costos

#### 5. Documentación
- ✅ README actualizado
- ✅ API docs
- ✅ Ejemplos de uso
- ✅ Troubleshooting guide

In [ ]:
# Ejemplo de deployment checklist
deployment_checklist = {
    "Testing": {
        "unit_tests": False,
        "integration_tests": False,
        "evaluation_dataset": False,
        "edge_cases": False
    },
    "Monitoring": {
        "langsmith": False,
        "logging": False,
        "alerts": False,
        "metrics": False
    },
    "Security": {
        "env_vars": False,
        "rate_limiting": False,
        "input_validation": False,
        "output_filtering": False
    }
}

def check_deployment_readiness(checklist: dict) -> dict:
    """Verifica si estás listo para deployment."""
    total_items = sum(len(items) for items in checklist.values())
    completed_items = sum(
        sum(1 for completed in items.values() if completed)
        for items in checklist.values()
    )
    
    readiness = completed_items / total_items
    
    print("🚀 Deployment Readiness Check\n")
    print("="*80)
    
    for category, items in checklist.items():
        print(f"\n{category}:")
        for item, completed in items.items():
            status = "✅" if completed else "❌"
            print(f"  {status} {item}")
    
    print("\n" + "="*80)
    print(f"Completado: {completed_items}/{total_items} ({readiness:.0%})")
    
    if readiness >= 0.8:
        print("\n✅ LISTO PARA DEPLOYMENT")
    elif readiness >= 0.5:
        print("\n⚠️ CASI LISTO - Completa items pendientes")
    else:
        print("\n❌ NO LISTO - Mucho trabajo pendiente")
    
    return {
        "readiness": readiness,
        "completed": completed_items,
        "total": total_items
    }

# Verificar
check_deployment_readiness(deployment_checklist)

## 💡 Ejercicio Final

In [ ]:
# Ejercicio: Crea un test suite completo para tu proyecto final
# Incluye:
# 1. Test cases para prompts
# 2. Evaluación de RAG (si aplica)
# 3. Testing de agentes (si aplica)
# 4. Deployment checklist

# 👉 Tu código aquí
print("Ejercicio: Test suite para proyecto final")

## 🎓 Resumen

### Conceptos Clave

1. **Testing de LLMs**: Diferente al testing tradicional
2. **Métricas**: Longitud, formato, relevancia, fidelidad
3. **LLM-as-a-Judge**: Usar LLMs para evaluar
4. **Testing de RAG**: Relevancia + Fidelidad
5. **Deployment**: Checklist completo

### Mejores Prácticas

- ✅ Temperatura=0 para tests
- ✅ Dataset de evaluación
- ✅ Múltiples métricas
- ✅ Monitoreo continuo
- ✅ Documentación completa

### Próxima Clase

**Demo Day**: ¡Presenta tu proyecto final!

---

**¡Excelente trabajo! 🚀**